# MFEGSN — Pipeline Colab (Marker + LangExtract)

Notebook optimisé pour une exécution **pas à pas** sur Google Colab.
Il sépare clairement les **blocs texte** (explications) et les **blocs code** (exécution).

**Ordre recommandé :** Étapes 1 → 10, avec tests optionnels si besoin.

## Étape 1 — Installer les dépendances
- Systèmes : zstd (requis pour Ollama).
- Python : marker-pdf, langextract, pillow.

In [ ]:
# Dépendances système
!apt-get update -qq
!apt-get install -y zstd ocrmypdf ghostscript -qq

# Dépendances Python
!python -m pip install -q --upgrade pip
!python -m pip install -q marker-pdf[full] langextract google-generativeai pillow PyPDF2 psutil

print("✅ Dépendances installées.")

## Étape 2 — Monter Google Drive
Exécutez cette cellule si vos PDF sont sur Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

# === CONFIGURATION DU CACHE SUR DRIVE ===
# Cela permet de sauvegarder les modèles (HuggingFace, Marker, etc.) 
# sur Drive pour ne pas les retélécharger à chaque session.

# Dossier de cache
DRIVE_CACHE_BASE = "/content/drive/MyDrive/.mfegsn_cache" 
DRIVE_CACHE_DIR = Path(DRIVE_CACHE_BASE)
DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Sous-dossiers
HF_CACHE_DRIVE = DRIVE_CACHE_DIR / "huggingface"
TORCH_CACHE_DRIVE = DRIVE_CACHE_DIR / "torch"
DATALAB_CACHE_DRIVE = DRIVE_CACHE_DIR / "datalab"

for cache_dir in [HF_CACHE_DRIVE, TORCH_CACHE_DRIVE, DATALAB_CACHE_DRIVE]:
    cache_dir.mkdir(parents=True, exist_ok=True)

# Variables d'environnement pour rediriger le cache
os.environ["HF_HOME"] = str(HF_CACHE_DRIVE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DRIVE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DRIVE)
os.environ["TORCH_HOME"] = str(TORCH_CACHE_DRIVE)
os.environ["XDG_CACHE_HOME"] = str(DRIVE_CACHE_DIR)

# Information
def get_dir_size(path):
    total = 0
    if path.exists():
        for f in path.rglob("*"):
            if f.is_file():
                total += f.stat().st_size
    return total / 1e9

cache_size = get_dir_size(DRIVE_CACHE_DIR)
print("="*60)
print("✅ Drive monté et Cache configuré")
print(f"📂 Cache: {DRIVE_CACHE_DIR}")
print(f"💾 Taille: {cache_size:.2f} GB")
print("="*60)

## Étape 3 — Configurer les dossiers
Modifiez les chemins ci-dessous selon votre Drive.

In [ ]:
from pathlib import Path
import shutil
import os

# === MODIFIEZ CES CHEMINS (Vers vos dossiers Drive) ===
DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/Géopolitique et Souveraineté Numériques/ALL/ALLPDF")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/Géopolitique et Souveraineté Numériques/ALL/ALLMD")

# === CONFIGURATION LOCALE (Optimisation Colab) ===
# On travaille en local pour éviter les lenteurs de Drive
INPUT_DIR = Path("/content/work/ALLPDF")
OUTPUT_DIR = Path("/content/work/ALLMD")

print("=" * 60)
print("📂 PRÉPARATION DE L'ENVIRONNEMENT DE TRAVAIL")
print("=" * 60)

# 1. Création des dossiers locaux
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = OUTPUT_DIR / "_FIGURES"
REFERENCES_DIR = OUTPUT_DIR / "_REFERENCES"
ANALYSES_DIR = OUTPUT_DIR / "_ANALYSES"
LOGS_DIR = OUTPUT_DIR / "_LOGS"
SEARCHABLE_DIR = OUTPUT_DIR / "_SEARCHABLE_PDF" 

for p in [FIGURES_DIR, REFERENCES_DIR, ANALYSES_DIR, LOGS_DIR, SEARCHABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# 2. Importation des PDFs depuis Drive
if DRIVE_INPUT_DIR.exists():
    print(f"📥 Copie des PDFs depuis : {DRIVE_INPUT_DIR}")
    pdfs = list(DRIVE_INPUT_DIR.glob("*.pdf"))
    print(f"   Nombre de fichiers identifiés : {len(pdfs)}")
    
    for i, pdf in enumerate(pdfs, 1):
        dest = INPUT_DIR / pdf.name
        # Copie si n'existe pas ou taille différente
        if not dest.exists() or dest.stat().st_size != pdf.stat().st_size:
            shutil.copy2(pdf, dest)
            if i % 10 == 0:
                print(f"   ... {i}/{len(pdfs)} copiés", end="\r")
    print(f"\n✅ Copie terminée. PDFs locaux : {len(list(INPUT_DIR.glob('*.pdf')))}")
else:
    print(f"⚠️ Dossier Drive introuvable : {DRIVE_INPUT_DIR}")

# Définition de la liste locale pour usage ultérieur
pdf_files = sorted(INPUT_DIR.glob("*.pdf"))
print(f"✅ Liste des fichiers PDF mise à jour : {len(pdf_files)} documents")

# 3. Préparation Dossiers Sortie Drive (Backup)
if not DRIVE_OUTPUT_DIR.exists():
    print(f"⚠️ Création du dossier sortie sur Drive : {DRIVE_OUTPUT_DIR}")
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print(f"📂 Entrée locale : {INPUT_DIR}")
print(f"📂 Sortie locale : {OUTPUT_DIR}")
print(f"☁️  Backup Drive : {DRIVE_OUTPUT_DIR}")
print("=" * 60)

## Étape 4 — (Optionnel) Ollama + Gemma 3 4B
Activez uniquement si vous utilisez LangExtract avec un modèle local.

In [ ]:
USE_OLLAMA = False  # Mettre True si vous voulez Gemma via Ollama

if USE_OLLAMA:
    import subprocess
    import time

    # Installer Ollama (si besoin)
    !curl -fsSL https://ollama.com/install.sh | sh

    # Démarrer Ollama en arrière-plan
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    print("⏳ Démarrage d'Ollama...")
    time.sleep(10)

    # Télécharger Gemma 3 4B
    print("📥 Téléchargement de Gemma 3 4B (≈3GB)...")
    !ollama pull gemma3:4b
    !ollama list
    print("✅ Gemma 3 4B prêt !")

## Étape 5 — Configurer Marker
Réglages optimisés (2 workers + extraction figures + références).

In [ ]:
import json
import re
import shutil
import base64
import psutil
import gc
from datetime import datetime
from pathlib import Path

import torch
from PIL import Image
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser

# --- A. CONFIGURATION ADAPTATIVE ---

def get_system_resources():
    """Détecte les ressources système disponibles (RAM, VRAM)."""
    ram = psutil.virtual_memory().total / (1024**3)  # GB
    vram = 0
    device_name = "CPU"
    
    if torch.cuda.is_available():
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) # GB
        device_name = torch.cuda.get_device_name(0)
    
    return {
        "ram_gb": ram,
        "vram_gb": vram,
        "device": device_name,
        "cuda": torch.cuda.is_available()
    }

resources = get_system_resources()
print(f"🖥️  Système détecté : {resources['device']}")
print(f"   RAM  : {resources['ram_gb']:.1f} GB")
print(f"   VRAM : {resources['vram_gb']:.1f} GB")

# Stratégie de configuration basée sur les ressources
if resources['ram_gb'] >= 24 and resources['vram_gb'] >= 15:
    # High-End (A100/L4 + High RAM)
    workers = 2
    batch_size = 4
    strategy = "Performance"
elif resources['ram_gb'] >= 12 and resources['vram_gb'] >= 14:
    # Mid-Range (T4 Standard)
    workers = 2  # IMPORTANT: workers=2 pour paralléliser
    batch_size = 2
    strategy = "Balanced"
else:
    # Low-End (CPU ou faible GPU)
    workers = 1
    batch_size = 1
    strategy = "Safe Mode"

marker_config = {
    "workers": workers,
    "extract_images": True,
    "images_as_base64": False,
    "use_llm": False,
    "force_ocr": True,
    "languages": ["fr", "en", "ar"],
    "paginate_output": True,
    "batch_size": batch_size,
}

print(f"⚙️  Stratégie activée : {strategy}")
print(f"   Workers : {marker_config['workers']}")
print(f"   Batch   : {marker_config['batch_size']}")
print(f"   OCR     : {marker_config['force_ocr']}")

if not resources['cuda']:
    print("⚠️ Attention : Exécution sur CPU détectée. Cela sera lent.")

print("📥 Chargement des modèles Marker...")

# Initialisation du convertisseur
model_dict = create_model_dict()
config_parser = ConfigParser(marker_config)
converter = PdfConverter(
    config=config_parser.generate_config_dict(),
    artifact_dict=model_dict,
)

print("✅ Marker configuré et prêt.")

## Étape 6 — Fonctions utilitaires
Extraction des références, figures et conversion PDF → Markdown.

In [ ]:
import subprocess
import os
import time

# --- GESTION MÉMOIRE ---

def clear_memory():
    """Libère la mémoire RAM et VRAM inutilisée."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_memory_stats():
    """Retourne l'utilisation actuelle de la mémoire."""
    mem = psutil.virtual_memory()
    ram_used = mem.percent
    vram_used = 0
    if torch.cuda.is_available():
        vram_used = torch.cuda.memory_reserved(0) / (1024**3) # GB
    return f"RAM: {ram_used}% | VRAM: {vram_used:.1f}GB"

# --- ANALYSE PDF ---

def get_pdf_complexity(pdf_path):
    """Calcule un score de complexité pour le tri."""
    try:
        size_mb = pdf_path.stat().st_size / (1024 * 1024)
        # Estimation rapide nombre de pages (via PyPDF2 si besoin, mais ici on évite d'ouvrir pour trier vite)
        # On utilise la taille comme proxy principal si on ne veut pas ouvrir tous les fichiers
        # Pour plus de précision, on peut utiliser pypdf :
        from pypdf import PdfReader
        try:
            with open(pdf_path, 'rb') as f:
                reader = PdfReader(f)
                num_pages = len(reader.pages)
        except:
            num_pages = size_mb * 2 # Fallback grossier
            
        return size_mb * num_pages
    except:
        return 0

def check_pdf_health(pdf_path, max_size_mb=50, max_pages=100):
    """Vérifie si le PDF nécessite un traitement spécial (division)."""
    warnings = []
    is_large = False
    
    try:
        size_mb = pdf_path.stat().st_size / (1024 * 1024)
        
        from pypdf import PdfReader
        with open(pdf_path, 'rb') as f:
            reader = PdfReader(f)
            if reader.is_encrypted:
                warnings.append("Encrypted")
            num_pages = len(reader.pages)
            
        if size_mb > max_size_mb or num_pages > max_pages:
            is_large = True
            warnings.append(f"Large PDF ({size_mb:.1f}MB, {num_pages} pages)")
            
    except Exception as e:
        warnings.append(f"Error reading: {str(e)}")
        
    return is_large, warnings

# --- SYNC / BACKUP ---

def sync_to_drive(local_dir, drive_dir, rsync=True):
    """Synchronise le dossier de sortie vers Google Drive."""
    print(f"🔄 Syncing {local_dir} -> {drive_dir}...")
    try:
        if rsync and shutil.which("rsync"):
            # Rsync est plus efficace (incrémental)
            cmd = ["rsync", "-av", "--update", f"{local_dir}/", f"{drive_dir}/"]
            subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
            print("✅ Rsync terminé.")
        else:
            # Fallback Python (plus lent car copie tout si on ne gère pas timestamps)
            # shutil.copytree est délicat pour update, on utilise dirs_exist_ok=True (Python 3.8+)
            shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
            print("✅ Copie terminйe (shutil).")
    except Exception as e:
        print(f"⚠️ Erreur sync : {e}")

# --- FONCTIONS EXISTANTES AMÉLIORÉES ---

def extract_references_from_markdown(markdown_text):
    """Extrait la section références/bibliographie du Markdown."""
    references = {
        "references_text": "",
        "references_list": [],
        "reference_count": 0,
    }

    ref_patterns = [
        r"(?i)(?:^|\n)#{1,3}\s*(references|références|bibliography|bibliographie|works\s*cited|sources?)\s*[\s:]*\n([\s\S]*?)(?=\n#{1,3}\s|\Z)",
        r"(?i)(?:^|\n)\*\*(references|références|bibliography|bibliographie)\*\*\s*[\s:]*\n([\s\S]*?)(?=\n\*\*|\n#{1,3}|\Z)",
    ]

    for pattern in ref_patterns:
        match = re.search(pattern, markdown_text, re.MULTILINE)
        if match:
            ref_section = match.group(2).strip()
            references["references_text"] = ref_section

            ref_lines = []
            lines = ref_section.split("\n")
            current_ref = ""

            for line in lines:
                line = line.strip()
                if not line:
                    if current_ref:
                        ref_lines.append(current_ref.strip())
                        current_ref = ""
                    continue

                if re.match(r"^(\[\d+\]|\d+\.|[-•]|\([A-Za-z]\))", line):
                    if current_ref:
                        ref_lines.append(current_ref.strip())
                    current_ref = line
                else:
                    current_ref += " " + line

            if current_ref:
                ref_lines.append(current_ref.strip())

            references["references_list"] = [r for r in ref_lines if len(r) > 20]
            references["reference_count"] = len(references["references_list"])
            break

    return references


def extract_figures_info(markdown_text, images_dict):
    """Extrait les informations sur les figures du document."""
    figures = []
    fig_pattern = r"(?i)(figure|fig\.)\s*(\d+)?\s*[:]?\s*(.{0,120})"

    for match in re.finditer(fig_pattern, markdown_text):
        title = (match.group(3) or "").strip()
        figures.append({
            "label": match.group(0).strip(),
            "title": title,
        })

    if images_dict:
        for img_name in images_dict.keys():
            existing = any(f.get("path") == img_name for f in figures)
            if not existing:
                figures.append({
                    "label": str(img_name),
                    "title": "",
                    "path": str(img_name),
                })

    return figures


def save_figures(images_dict, doc_name, figures_base_folder):
    """Sauvegarde les figures extraites dans un dossier dédié."""
    if not images_dict:
        return []

    doc_figures_folder = figures_base_folder / doc_name
    doc_figures_folder.mkdir(parents=True, exist_ok=True)
    saved_paths = []

    for img_name, img_data in images_dict.items():
        safe_name = re.sub(r"[^a-zA-Z0-9_-]+", "_", str(img_name))
        img_path = doc_figures_folder / f"{safe_name}.png"

        try:
            if isinstance(img_data, Image.Image):
                img_data.save(img_path)
            elif isinstance(img_data, (bytes, bytearray)):
                with open(img_path, "wb") as f:
                    f.write(img_data)
            elif isinstance(img_data, str):
                if img_data.startswith("data:image"):
                    b64_data = img_data.split(",", 1)[1]
                    with open(img_path, "wb") as f:
                        f.write(base64.b64decode(b64_data))
                elif Path(img_data).exists():
                    shutil.copy(img_data, img_path)
                else:
                    with open(img_path, "wb") as f:
                        f.write(base64.b64decode(img_data))
            else:
                continue

            saved_paths.append(str(img_path))
        except Exception:
            continue

    return saved_paths

def markdown_to_ocr_text(markdown_path: Path, output_txt: Path):
    """Convertit le Markdown Marker en texte brut pour OCRmyPDF."""
    try:
        with open(markdown_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Nettoyage basique
        content = re.sub(r'!\[.*?\]\(.*?\)', '', content)
        content = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', content)
        content = re.sub(r'```.*?```', '', content, flags=re.DOTALL)
        content = re.sub(r'#{1,6}\s+', '', content)
        
        with open(output_txt, 'w', encoding='utf-8') as f:
            f.write(content)
        return True
    except Exception as e:
        print(f"⚠️ Erreur conversion MD→TXT : {e}")
        return False

def make_searchable_pdf(src_pdf: Path, out_pdf: Path, sidecar_txt: Path = None):
    """Génère un PDF consultable (Searchable PDF) via OCRMyPDF (Robuste)."""
    max_retries = 2
    for attempt in range(max_retries + 1):
        try:
            cmd = [
                "ocrmypdf",
                "--optimize", "1",
                "--jobs", "2",
                "--output-type", "pdf",
                "--language", "fra+eng+ara", # Multilangue
                "--skip-big", "10", # Skip pages très lourdes en images si besoin
                "--tesseract-timeout", "300" # Timeout par page
            ]
            
            if sidecar_txt and sidecar_txt.exists():
                cmd.extend(["--sidecar", str(sidecar_txt)])
            else:
                cmd.append("--skip-text") 
            
            cmd.extend([str(src_pdf), str(out_pdf)])
            
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=600) # Timeout global process
            return True, None
            
        except subprocess.TimeoutExpired:
            print(f"⚠️ Timeout OCR (essai {attempt+1}/{max_retries})")
            clear_memory()
            continue
        except subprocess.CalledProcessError as e:
            err_msg = e.stderr.decode()
            if "Out of memory" in err_msg or "memory" in err_msg.lower():
                print(f"⚠️ OOM OCR (essai {attempt+1}/{max_retries}) -> Nettoyage mémoire")
                clear_memory()
                continue
            return False, err_msg
        except FileNotFoundError:
            return False, "OCRMyPDF non trouvé"
        except Exception as e:
            return False, str(e)
            
    return False, "Echec après plusieurs tentatives"

def convert_pdf_complete(pdf_path, doc_name):
    """Conversion simple d'un PDF (Wrapper existant)."""
    # Note: Cette fonction est conservée pour compatibilité mais sera peu utilisée 
    # si le pipeline principal gère la logique.
    
    result_data = {
        "doc_name": doc_name,
        "markdown_path": "",
        "figures": [],
        "figures_paths": [],
        "references": {},
        "searchable_pdf": "",
        "error": None,
    }

    try:
        result = converter(str(pdf_path))
        markdown_text = getattr(result, "markdown", "") or ""
        images_dict = getattr(result, "images", {}) or {}

        md_path = OUTPUT_DIR / f"{doc_name}.md"
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(markdown_text)

        result_data["markdown_path"] = str(md_path)
        result_data["figures_paths"] = save_figures(images_dict, doc_name, FIGURES_DIR)
        result_data["figures"] = extract_figures_info(markdown_text, images_dict)
        result_data["references"] = extract_references_from_markdown(markdown_text)
        
        return result_data
    except Exception as e:
        result_data["error"] = str(e)
        return result_data

## Étape 7 — LangExtract (optionnel)
Activez uniquement si vous souhaitez l'extraction structurée.

In [ ]:
USE_LANGEXTRACT = False  # Mettre True pour activer LangExtract

PROMPT_TEMPLATE = """
Vous êtes un assistant d'analyse pour des documents en sciences sociales.
Retournez un JSON structuré avec les sections suivantes :

1. CONTEXTE
- Thème principal
- Zone géographique
- Période

2. ACTEURS
- Institutions
- Pays
- Organisations

3. CONCEPTS CLÉS
- Mots-clés
- Concepts

4. DONNÉES
- Chiffres clés (si disponibles)

5. RÉFÉRENCES
- Principales références citées

6. FIGURES ET TABLEAUX
- Liste des figures mentionnées

Répondez uniquement avec un JSON valide.
"""

def _safe_json(obj):
    try:
        return json.loads(json.dumps(obj))
    except Exception:
        return {"raw": str(obj)}


def extract_with_langextract(markdown_text, doc_name, references_data=None, figures_data=None):
    """Extraction structurée avec LangExtract (optionnel)."""
    if not USE_LANGEXTRACT:
        return {"status": "skipped", "reason": "USE_LANGEXTRACT=False"}

    enriched_text = markdown_text

    if references_data and references_data.get("reference_count", 0) > 0:
        enriched_text += "\n\n## RÉFÉRENCES\n"
        enriched_text += f"Nombre de références : {references_data['reference_count']}\n"
        for i, ref in enumerate(references_data.get("references_list", [])[:20], 1):
            enriched_text += f"[{i}] {ref}\n"

    if figures_data:
        enriched_text += "\n\n## FIGURES IDENTIFIÉES\n"
        enriched_text += f"Nombre de figures : {len(figures_data)}\n"
        for fig in figures_data[:10]:
            enriched_text += f"- {fig.get('label', '')} {fig.get('title', '')}\n"

    try:
        import langextract as lx
        if hasattr(lx, "extract"):
            extraction = lx.extract(enriched_text, prompt=PROMPT_TEMPLATE)
        elif hasattr(lx, "LangExtract"):
            extractor = lx.LangExtract()
            extraction = extractor.extract(enriched_text, prompt=PROMPT_TEMPLATE)
        else:
            return {"status": "error", "error": "API LangExtract introuvable"}

        return _safe_json(extraction)
    except Exception as e:
        return {"status": "error", "error": str(e)}

## Étape 8 — Test sur un PDF (optionnel)
Permet de valider la configuration avant le batch.

In [ ]:
if pdf_files:
    sample_path = pdf_files[0]
    sample_name = sample_path.stem
    print(f"🧪 Test de conversion sur : {sample_name}")
    print(f"📊 Taille : {sample_path.stat().st_size / (1024*1024):.1f} MB")

    # Vérification santé et choix de la méthode
    is_large, warnings = check_pdf_health(sample_path)
    
    if is_large:
        print(f"⚠️ PDF volumineux détecté ({', '.join(warnings)}).")
        print("⚡ Passage automatique en mode 'Chunking' pour éviter le crash mémoire.")
        sample_result = process_large_pdf(sample_path, sample_name)
    else:
        print("✅ PDF de taille standard. Traitement direct.")
        try:
             sample_result = convert_pdf_complete(sample_path, sample_name)
        except Exception as e:
             if "memory" in str(e).lower():
                 print("🚨 OOM détecté en mode standard. Tentative de reprise en mode Chunking...")
                 clear_memory()
                 sample_result = process_large_pdf(sample_path, sample_name)
             else:
                 sample_result = {"error": str(e)}

    # Affichage résultat succinct
    print("\n--- Résultat du Test ---")
    if sample_result.get("error"):
        print(f"❌ Erreur : {sample_result['error']}")
    else:
        print(f"✅ Succès !")
        print(f"📄 Pages traitées (approx) : {sample_result.get('references', {}).get('reference_count', 0)}")
        print(f"💾 Sortie : {sample_result.get('markdown_path')}")
else:
    print("Aucun PDF trouvé dans le dossier d'entrée.")

## Étape 9 — Pipeline complet avec reprise
Traitement batch + logs + reprise automatique.

## Étape 8b — Gestion intelligente des gros PDFs

### Division et fusion automatiques pour optimiser la mémoire

Cette section implémente une logique de "Divide & Conquer" pour traiter les documents volumineux qui causeraient des erreurs de mémoire (OOM) s'ils étaient traités en une seule fois.

**Logique :**
1. **Détection** : Si PDF > 100MB ou > 100 pages.
2. **Division** : Découpage en chunks de 100 pages.
3. **Traitement** : Chaque chunk est converti indépendamment.
4. **Fusion** : Reconstitution du Markdown final, du PDF Searchable et des métadonnées.

In [ ]:
import sys

# Gestion robuste des imports pypdf / PyPDF2
# On privilégie PyPDF2 qui est plus stable pour PdfMerger
try:
    from PyPDF2 import PdfReader, PdfWriter, PdfMerger
    print("📚 Utilisation de PyPDF2 (PdfMerger)")
except ImportError:
    try:
        # Fallback: anciennes versions de PyPDF2
        from PyPDF2 import PdfFileReader as PdfReader, PdfFileWriter as PdfWriter, PdfFileMerger as PdfMerger
        print("📚 Utilisation de PyPDF2 (PdfFileMerger - ancienne API)")
    except ImportError:
        # Dernier recours: pypdf récent (mais peut ne pas avoir PdfMerger)
        from pypdf import PdfReader, PdfWriter
        try:
            from pypdf import PdfMerger
        except ImportError:
            from pypdf import PdfFileMerger as PdfMerger
        print("📚 Utilisation de pypdf")

def split_pdf(pdf_path, chunk_size=25):
    """Divise un PDF en chunks de taille donnée."""
    chunks = []
    temp_dir = pdf_path.parent / "temp_chunks" / pdf_path.stem
    temp_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        reader = PdfReader(str(pdf_path)) # str() pour compatibilité anciens pypdf
        try:
            total_pages = len(reader.pages)
        except AttributeError:
            total_pages = reader.getNumPages() # Fallback v2
        
        for i in range(0, total_pages, chunk_size):
            writer = PdfWriter()
            end = min(i + chunk_size, total_pages)
            
            for page_num in range(i, end):
                try:
                   page = reader.pages[page_num]
                except AttributeError:
                   page = reader.getPage(page_num) # Fallback v2
                
                writer.add_page(page)
            
            chunk_name = f"{pdf_path.stem}_chunk_{i//chunk_size + 1:03d}.pdf"
            chunk_path = temp_dir / chunk_name
            
            with open(chunk_path, "wb") as f:
                writer.write(f)
            chunks.append(chunk_path)
            
        return chunks, temp_dir
    except Exception as e:
        print(f"⚠️ Erreur division PDF : {e}")
        return [], None

def merge_markdowns(chunks_data, final_doc_name):
    """Fusionne les Markdowns des chunks."""
    full_md = ""
    # Header global (optionnel)
    
    for i, data in enumerate(chunks_data):
        chunk_md = ""
        if 'markdown_path' in data and Path(data['markdown_path']).exists():
            with open(data['markdown_path'], 'r', encoding='utf-8') as f:
                chunk_md = f.read()
        
        # Ajouter séparateur visible
        full_md += f"\n\n<!-- CHUNK {i+1} START -->\n\n"
        full_md += chunk_md
        full_md += f"\n\n<!-- CHUNK {i+1} END -->\n\n"
    
    final_md_path = OUTPUT_DIR / f"{final_doc_name}.md"
    with open(final_md_path, 'w', encoding='utf-8') as f:
        f.write(full_md)
        
    return str(final_md_path)

def merge_searchable_pdfs(chunks_pdfs, output_path):
    """Fusionne les PDFs Searchable."""
    merger = PdfMerger()
    try:
        valid_pdfs = [p for p in chunks_pdfs if p and Path(p).exists()]
        if not valid_pdfs:
             return False
             
        for pdf in valid_pdfs:
            merger.append(str(pdf))
        
        merger.write(str(output_path))
        merger.close()
        return True
    except Exception as e:
        print(f"⚠️ Erreur fusion PDF : {e}")
        return False

def process_single_chunk(chunk_path, chunk_name):
    """Traite un chunk individuel (Marker + OCR)."""
    # 1. Conversion Marker
    result_data = convert_pdf_complete(chunk_path, chunk_name)
    
    # 2. OCR Searchable PDF
    chunk_pdf_out = SEARCHABLE_PDF_DIR / f"{chunk_name}_ocr.pdf"
    chunk_txt_sidecar = OUTPUT_DIR / f"{chunk_name}_ocr_sidecar.txt"
    
    # 2b. Check error before continuing
    if result_data.get('error'):
        return result_data

    # Préparer le texte pour aider l'OCR (sidecar)
    if result_data.get('markdown_path') and markdown_to_ocr_text(Path(result_data['markdown_path']), chunk_txt_sidecar):
         success, _ = make_searchable_pdf(chunk_path, chunk_pdf_out, sidecar_txt=chunk_txt_sidecar)
    else:
         success, _ = make_searchable_pdf(chunk_path, chunk_pdf_out)
         
    result_data['searchable_pdf'] = str(chunk_pdf_out) if success else None
    
    # Clean memory after big operation
    clear_memory()
    
    return result_data

def process_large_pdf(pdf_path, doc_name, chunk_size=25):
    """Orchestre le découpage, traitement et fusion d'un gros PDF."""
    print(f"🐘 Traitement Gros PDF : {doc_name}")
    
    # 1. Split
    chunks, temp_dir = split_pdf(pdf_path, chunk_size)
    if not chunks:
        return {"error": "Split failed"}
        
    print(f"✂️  Découpé en {len(chunks)} parties.")
    
    chunk_results = []
    chunk_pdfs = []
    processing_error = False
    
    # 2. Process Loop
    for i, chunk in enumerate(chunks):
        print(f"   ► Chunk {i+1}/{len(chunks)} ({chunk.name})...")
        try:
            res = process_single_chunk(chunk, chunk.stem)
            
            if res.get('error'):
                 print(f"     ❌ Erreur Marker sur ce chunk : {res['error']}")
                 # On continue quand même pour essayer de sauver les autres parties ? 
                 # Oui, mais on note l'erreur.
            
            chunk_results.append(res)
            chunk_pdfs.append(res.get('searchable_pdf'))
        except Exception as e:
            print(f"\n   ❌ Exception Chunk {i+1}: {e}")
            processing_error = True
            
    print(f"\n✅ Tous les chunks traités.")
    
    # 3. Merge
    merge_success = False
    final_md_path = None
    final_pdf_path = None
    
    if chunk_results: # Tentative de fusion même si erreurs partielles
        try:
            # MD
            final_md_path = merge_markdowns(chunk_results, doc_name)
            
            # PDF
            final_pdf_path = SEARCHABLE_PDF_DIR / f"{doc_name}_searchable.pdf"
            merge_success = merge_searchable_pdfs(chunk_pdfs, final_pdf_path)
        except Exception as e:
            print(f"⚠️ Erreur Fusion finale : {e}")
    
    # Figures (Consolidation)
    all_figures = []
    for res in chunk_results:
        all_figures.extend(res.get('figures', []))
        
    # References (Deduplicate)
    all_refs = set()
    for res in chunk_results:
        refs = res.get('references', {}).get('references_list', [])
        for r in refs:
            all_refs.add(r)
            
    # Cleanup Temp (seulement si succès total)
    if merge_success and not processing_error:
        try:
            shutil.rmtree(temp_dir)
            # Supprimer aussi les fichiers intermédiaires (chunks pdf/md dans output)
            for res in chunk_results:
                try: 
                    if res.get('markdown_path'): Path(res['markdown_path']).unlink(missing_ok=True)
                    if res.get('searchable_pdf'): Path(res['searchable_pdf']).unlink(missing_ok=True)
                except: pass
        except:
            pass
    else:
        print(f"⚠️ Fusion incomplète ou erreurs : Chunks conservés dans {temp_dir}")
        
    return {
        "doc_name": doc_name,
        "markdown_path": final_md_path,
        "searchable_pdf": str(final_pdf_path) if merge_success else None,
        "figures": all_figures,
        "references": {"references_list": list(all_refs), "reference_count": len(all_refs)},
        "chunked": True,
        "error": "Partial/Total failure" if processing_error else None
    }

In [ ]:
import time
import pandas as pd
from IPython.display import display, HTML

# Dossier pour les rapports
LOGS_DIR = OUTPUT_BASE_DIR / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

report_file = LOGS_DIR / f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
checkpoint_file = LOGS_DIR / "checkpoint.json"

# Charger checkpoint
processed_files = set()
if checkpoint_file.exists():
    try:
        with open(checkpoint_file, "r") as f:
            processed_files = set(json.load(f))
        print(f"🔄 Reprise : {len(processed_files)} fichiers déjà traités.")
    except:
        print("⚠️ Checkpoint corrompu, redémarrage.")

# 1. Tri Intelligent des Fichiers
print("📊 Analyse et tri des documents...")
pdf_files_ranked = []
for p in pdf_files:
    pdf_files_ranked.append((p, get_pdf_complexity(p)))

# Trier : les plus petits d'abord (strategy='easy_first') ou inversement
# On traite les petits d'abord pour avoir des résultats rapides
pdf_files_ranked.sort(key=lambda x: x[1])

# Afficher les extrêmes
print(f"   Plus complexe : {pdf_files_ranked[-1][0].name} (Score: {pdf_files_ranked[-1][1]:.0f})")
print(f"   Moins complexe : {pdf_files_ranked[0][0].name} (Score: {pdf_files_ranked[0][1]:.0f})")

pdf_queue = [x[0] for x in pdf_files_ranked]

results_list = []
start_time = time.time()
CHECKPOINT_INTERVAL = 5

print(f"🚀 Démarrage du pipeline sur {len(pdf_queue)} documents.")
print(f"   Mémoire Initiale : {get_memory_stats()}")

for i, pdf_path in enumerate(pdf_queue):
    doc_name = pdf_path.stem
    
    if doc_name in processed_files:
        continue

    iter_start = time.time()
    print(f"\n📄 [{i+1}/{len(pdf_queue)}] Traitement : {doc_name}")
    
    # 2. Gestion Mémoire Préventive
    clear_memory()
    
    # 3. Check Health
    is_large, warnings = check_pdf_health(pdf_path)
    if warnings:
        print(f"   ⚠️ {', '.join(warnings)}")

    try:
        # A. TRAITEMENT ADAPTATIF
        if is_large:
            # Mode "Gros PDF" (Division + Fusion)
            result_data = process_large_pdf(pdf_path, doc_name)
        else:
            # Mode Standard (Direct)
            result_data = convert_pdf_complete(pdf_path, doc_name)
            
            # Ajout explicite de l'OCR si pas d'erreur
            if not result_data.get("error"):
                out_pdf = SEARCHABLE_PDF_DIR / f"{doc_name}_searchable.pdf"
                
                # Sidecar text pour guider l'OCR
                sidecar_txt = OUTPUT_DIR / f"{doc_name}_sidecar.txt"
                markdown_to_ocr_text(Path(result_data['markdown_path']), sidecar_txt)
                
                print("   🔍 OCRmyPDF en cours...")
                ok, err = make_searchable_pdf(pdf_path, out_pdf, sidecar_txt=sidecar_txt)
                if ok:
                    result_data["searchable_pdf"] = str(out_pdf)
                else:
                    print(f"   ❌ Erreur OCR : {err}")

        # B. ENRICHISSEMENT (LANGEXTRACT)
        if not result_data.get("error") and USE_LANGEXTRACT:
            print("   🧠 Extraction structurée (LangExtract)...")
            md_content = ""
            if Path(result_data["markdown_path"]).exists():
                with open(result_data["markdown_path"], "r") as f:
                    md_content = f.read()
            
            extraction = extract_with_langextract(
                md_content, 
                doc_name, 
                result_data.get("references"), 
                result_data.get("figures")
            )
            
            json_path = OUTPUT_DIR / f"{doc_name}_structure.json"
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(extraction, f, ensure_ascii=False, indent=2)
            result_data["extraction_json"] = str(json_path)

        # C. LOGGING
        duration = time.time() - iter_start
        status = "SUCCESS" if not result_data.get("error") else "ERROR"
        
        log_entry = {
            "doc_name": doc_name,
            "status": status,
            "duration": round(duration, 2),
            "pages": result_data.get("references", {}).get("reference_count", 0), # Proxy
            "error": result_data.get("error"),
            "path": str(pdf_path)
        }
        results_list.append(log_entry)
        
        print(f"   ⏱️ {duration:.1f}s | Status: {status}")
        if status == "ERROR":
            print(f"   🔴 {result_data['error']}")
        
        # D. CHECKPOINT & SYNC
        processed_files.add(doc_name)
        if len(results_list) % CHECKPOINT_INTERVAL == 0:
            # Sauvegarde CSV partiel
            pd.DataFrame(results_list).to_csv(report_file, index=False)
            # Sauvegarde Checkpoint
            with open(checkpoint_file, "w") as f:
                json.dump(list(processed_files), f)
            # Sync Drive
            sync_to_drive(OUTPUT_BASE_DIR, DRIVE_OUTPUT_DIR)
            
            print(f"   💾 Checkpoint & Sync effectués ({get_memory_stats()})")

    except Exception as e:
        print(f"   🚨 EXCEPTION CRITIQUE : {e}")
        time.sleep(5) # Pause cool-down
        clear_memory()

# Rapport Final
df_results = pd.DataFrame(results_list)
df_results.to_csv(report_file, index=False)
try:
    display(df_results)
except:
    pass

print(f"\n🏁 Traitement terminé en {(time.time() - start_time)/60:.1f} minutes.")

## Étape 10 — Lancer le traitement
Décommentez si nécessaire, puis exécutez.

In [ ]:
results, errors = process_all_documents()
export_all_results(results)

print("✅ Terminé.")
if errors:
    print(f"⚠️  Erreurs : {len(errors)}")

## Étape 11 — Sauvegarder sur Google Drive
Copie des résultats depuis l'environnement local vers votre Drive pour stockage permanent.

In [ ]:
# Synchronisation Finale
print("="*60)
print(f"🚀 SYNCHRONISATION FINALE VERS DRIVE")
print("="*60)
print(f"📂 Source :      {OUTPUT_BASE_DIR}")
print(f"☁️  Destination : {DRIVE_OUTPUT_DIR}")

# Utilise la fonction robuste définie précédemment
sync_to_drive(OUTPUT_BASE_DIR, DRIVE_OUTPUT_DIR)

# Résumé
try:
    md_count = len(list(DRIVE_OUTPUT_DIR.glob("**/*.md")))
    pdf_count = len(list(DRIVE_OUTPUT_DIR.glob("**/*.pdf")))
    print(f"\n📊 Bilan sur Drive :")
    print(f"   - Markdowns : {md_count}")
    print(f"   - PDFs      : {pdf_count}")
except:
    pass

print("✅ Traitement terminé.")